# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Use CUDA async allocator to reduce fragmentation:
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# Suppress TensorFlow logging (0: ALL, 1: INFO, 2: WARNING, 3: ERROR):
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# # If it fails to determine best cudnn convolution algorithm
# os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [ ]:
# # Disable all auto-JIT clustering at the process level
# os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [ ]:
get_gpu_info()

## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 2000
EPOCHS = 50

SAMPLER_SEED = 0

STEPS_PER_EXECUTION = 8

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
USE_JIT_COMPILE = True

In [ ]:
# Number of top trials to save
TOP_K = 3

# Order to rank trials by:
# "ascending" -> the lowest value is the best
# "descending" -> the highest value is the best
ORDER = "descending"

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "s009_accuracy"

# Direction of optimization:
# "minimize" -> the lowest value is the best
# "maximize" -> the highest value is the best
DIRECTION = "minimize"

In [ ]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [ ]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/nas_1")

## 3. Data Loading and Preprocessing

In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels(s008_path="./data/s008", s009_path="./data/s009")

In [ ]:
# Load LOS/NLOS raw subsets and build condition-specific splits
DATA_ROOT = Path("/media/matheus/SSD-2/matheus/datasets/RayWise")

S008_COORD_CSV = DATA_ROOT / "Raymobtime_s008/raw_data/CoordVehiclesRxPerScene_s008.csv"
S008_LIDAR_FOLDER = DATA_ROOT / "Raymobtime_s008/processed_raw_data/lidar_data_s008"
S008_BEAM_OUTPUT = DATA_ROOT / "Raymobtime_s008/baseline_data/beam_output/beams_output_s008.npz"

S009_COORD_CSV = DATA_ROOT / "Raymobtime_s009/raw_data/CoordVehiclesRxPerScene_s009.csv"
S009_LIDAR_FOLDER = DATA_ROOT / "Raymobtime_s009/processed_raw_data/lidar_data_s009"
S009_BEAM_OUTPUT = DATA_ROOT / "Raymobtime_s009/baseline_data/beam_output/beams_output_test.npz"

(
    (x_lidar_s008_los, x_coord_s008_los, y_s008_los),
    (x_lidar_s009_los, x_coord_s009_los, y_s009_los),
    (x_lidar_s008_nlos, x_coord_s008_nlos, y_s008_nlos),
    (x_lidar_s009_nlos, x_coord_s009_nlos, y_s009_nlos),
) = load_dataset_raw_sparse_labels_by_condition(
    s008_coord_csv=str(S008_COORD_CSV),
    s008_lidar_folder=str(S008_LIDAR_FOLDER),
    s008_beam_output_path=str(S008_BEAM_OUTPUT),
    s009_coord_csv=str(S009_COORD_CSV),
    s009_lidar_folder=str(S009_LIDAR_FOLDER),
    s009_beam_output_path=str(S009_BEAM_OUTPUT),
    data_seed=0,
    report_label_coverage=True,
)

# print shapes
print("\n\n\n")
print("s008 LOS:", x_lidar_s008_los.shape, x_coord_s008_los.shape, y_s008_los.shape)
print("s008 NLOS:", x_lidar_s008_nlos.shape, x_coord_s008_nlos.shape, y_s008_nlos.shape)
print("s009 LOS:", x_lidar_s009_los.shape, x_coord_s009_los.shape, y_s009_los.shape)
print("s009 NLOS:", x_lidar_s009_nlos.shape, x_coord_s009_nlos.shape, y_s009_nlos.shape)

## Hyperparameters

In [ ]:
kparams = KParams(
    activation_choices={
        "relu": tf.keras.activations.relu,
        "gelu": tf.keras.activations.gelu,
        "silu": tf.keras.activations.silu,
        "elu": tf.keras.activations.elu,
        "sigmoid": tf.keras.activations.sigmoid,
        "tanh": tf.keras.activations.tanh,
        "none": None,
    },
    regularizer_choices={
        # "l2": tf.keras.regularizers.l2,
        "none": None,
    },
    optimizer_choices={
        # "sgd": tf.keras.optimizers.SGD(momentum=0.9),
        # "adam": tf.keras.optimizers.Adam(),
        "adamw": tf.keras.optimizers.AdamW(weight_decay=1e-4),
        # "lion": tf.keras.optimizers.Lion(beta_1=0.9, beta_2=0.99),
        # "rmsprop": tf.keras.optimizers.RMSprop(),
    },
    # scaler_choices={
    #     "standard": StandardScaler,
    #     "minmax_0_1": lambda: MinMaxScaler(feature_range=(0, 1)),
    #     "minmax_-1_1": lambda: MinMaxScaler(feature_range=(-1, 1)),
    # },
    learning_rate=(1e-4, 1e-2),
)

## 5. Model Definition

In [ ]:
def build_model(
    trial: optuna.Trial,
    kparams: dict,
    *,
    show_summary: bool = True,
    **kwargs: Any,
) -> tf.keras.Model:

    train_seed = kwargs.get("train_seed")

    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=train_seed,
    )

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    num_conv_layers = trial.suggest_int("num_conv_layers", 1, 2)

    for i in range(num_conv_layers):
        x = build_cnn1d(
            trial=trial,
            kparams=kparams,
            x=combined if i == 0 else x,  # Use combined only for the first layer
            name_prefix=f"conv1d_{i}",
            # Filters
            filters_range=trial.suggest_categorical(f"conv1d_{i}_filters", [64, 128, 256, 512, 1024]),
            # filters_step=40,
            # Kernel size
            kernel_size_range=(1, 9),
            kernel_size_step=1,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_{i}_strides", 1, 2),
            kernel_initializer=initializer,
        )
        #! pool size = 1 means no downsampling
        pool_size = trial.suggest_int(f"pool_size_{i}", 1, 4, step=1)
        x = layers.MaxPooling1D(pool_size=pool_size, name=f"max_pool_{i}")(x)

    pooling_type = trial.suggest_categorical("pooling_type", ["flatten", "max", "average"])
    if pooling_type == "flatten":
        x = layers.Flatten(name="flatten_cnn_output")(x)
    elif pooling_type == "max":
        x = layers.GlobalMaxPooling1D(name="global_max_pooling")(x)
    else:
        x = layers.GlobalAveragePooling1D(name="global_avg_pooling")(x)

    num_dense_layers = trial.suggest_int("num_dense_layers", 0, 1)
    for i in range(num_dense_layers):
        x = build_dnn(
            trial=trial,
            kparams=kparams,
            x=x,
            name_prefix=f"dense_{i}",
            units_range=(25, 500),
            units_step=25,
            dropout_rate_range=(0.0, 0.5),
            dropout_rate_step=0.1,
            kernel_initializer=initializer,
        )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=USE_JIT_COMPILE,  # For XLA speedup, does not support determinism
        steps_per_execution=STEPS_PER_EXECUTION,
    )

    return model

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    **kwargs: Any,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.
    """
    (print(f"Running trial {trial.number}..."), clear_session())

    DATA_SEED = 0
    TRAIN_SEED = 0

    set_random_seed(TRAIN_SEED)

    global s009_coord_input, s009_lidar_input, s009_y
    global s008_coord_input, s008_lidar_input, s008_y_train

    global x_lidar_s008_nlos, x_coord_s008_nlos, y_s008_nlos
    global x_lidar_s009_nlos, x_coord_s009_nlos, y_s009_nlos

    (
        x_lidar_train,
        x_lidar_val,
        x_coord_train,
        x_coord_val,
        y_train,
        y_val,
    ) = train_test_split(
        x_lidar_s008_nlos,
        x_coord_s008_nlos,
        y_s008_nlos,
        test_size=0.2,
        random_state=DATA_SEED,
        shuffle=True,
    )

    backup_dir = kwargs["backup"]
    model_dir = kwargs["model"]
    fig_dir = kwargs["fig"]
    tensorboard_dir = kwargs["tensorboard"]
    logs_dir = kwargs["logs"]
    history_dir = kwargs["history"]
    scaler_dir = kwargs["scaler"]

    try:
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train).astype(np.float32)
        x_coord_val = coord_scaler.transform(x_coord_val).astype(np.float32)
        s009_coord_input = coord_scaler.transform(s009_coord_input).astype(np.float32)
        s008_coord_input = coord_scaler.transform(s008_coord_input).astype(np.float32)
        x_coord_s008_nlos = coord_scaler.transform(x_coord_s008_nlos).astype(np.float32)
        x_coord_s009_nlos = coord_scaler.transform(x_coord_s009_nlos).astype(np.float32)

        scaler_path = os.path.join(scaler_dir, f"trial_{trial.number}.pkl")
        with open(scaler_path, "wb") as scaler_file:
            pickle.dump(coord_scaler, scaler_file)

        model = build_model(
            trial=trial,
            kparams=kparams,
            show_summary=False,
            train_seed=TRAIN_SEED,
        )
        BATCH_SIZE = 64

        prune_model_by_config(
            trial=trial,
            model=model,
            thresholds={
                "model_size": 350,
                "memory_mb": 9000,
                "param": 2e8,
                "flops": 1.2e10,
            },
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
        )

        history = model.fit(
            x=[x_lidar_train, x_coord_train],
            y=y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks_study(
                trial=trial,
                monitor="val_loss",
            ),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————— Model characteristics —————————————————————————— #
        set_user_attr_model_stats(
            trial=trial,
            model=model,
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
            test_runs=10,
            device="gpu/0",
            stats_to_measure=(
                "parameters",
                "model_size",
                "flops",
                "macs",
                "summary",
                "inference_latency",
                # "cpu_util_percent",
                # "cpu_power_rapl_w",
                # "ram_used_bytes",
                # "ram_util_percent",
                # "gpu_util_percent",
                # "gpu_mem_used_bytes",
                # "gpu_power_w",
            ),
            extra_attrs=None,
            verbose=1,
        )

        # Evaluate on full s009
        s009_loss, s009_acc = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=BATCH_SIZE, verbose=2
        )

        # Evaluate on s008
        s008_loss, s008_acc = model.evaluate(
            [s008_lidar_input, s008_coord_input], s008_y_train, batch_size=BATCH_SIZE, verbose=2
        )

        # Evaluate on s008 NLOS
        s008_nlos_loss, s008_nlos_acc = model.evaluate(
            [x_lidar_s008_nlos, x_coord_s008_nlos], y_s008_nlos, batch_size=BATCH_SIZE, verbose=2
        )

        # Evaluate on s009 NLOS
        s009_nlos_loss, s009_nlos_acc = model.evaluate(
            [x_lidar_s009_nlos, x_coord_s009_nlos], y_s009_nlos, batch_size=BATCH_SIZE, verbose=2
        )

        best_idx = int(np.argmax(history.history["val_accuracy"]))

        best_train_loss = float(history.history["loss"][best_idx])
        best_val_loss = float(history.history["val_loss"][best_idx])
        best_train_acc = float(history.history["accuracy"][best_idx])
        best_val_acc = float(history.history["val_accuracy"][best_idx])

        trial.set_user_attr("best_epoch", best_idx + 1)
        trial.set_user_attr("best_train_loss", best_train_loss)
        trial.set_user_attr("best_val_loss", best_val_loss)
        trial.set_user_attr("s008_loss", s008_loss)
        trial.set_user_attr("s009_loss", s009_loss)
        trial.set_user_attr("s008_nlos_loss", s008_nlos_loss)
        trial.set_user_attr("s009_nlos_loss", s009_nlos_loss)

        trial.set_user_attr("best_train_accuracy", best_train_acc)
        trial.set_user_attr("best_val_accuracy", best_val_acc)
        trial.set_user_attr("s008_accuracy", s008_acc)
        trial.set_user_attr("s009_accuracy", s009_acc)
        trial.set_user_attr("s008_nlos_accuracy", s008_nlos_acc)
        trial.set_user_attr("s009_nlos_accuracy", s009_nlos_acc)

        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
            "train_accuracy": history.history["accuracy"],
            "val_accuracy": history.history["val_accuracy"],
        }

        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        if len(history.history["val_loss"]) > 1:
            report_cross_validation_scores(trial, scores=history.history["val_loss"])

        return best_val_loss
    except ValueError as e:
        if "Negative dimension size" in str(e):
            raise optuna.TrialPruned("Pruned, invalid pooling config") from e
        raise
    except Exception as e:
        log_trial_error(
            trial=trial,
            exc=e,
            logs_dir=logs_dir,
            prune_on={
                tf.errors.ResourceExhaustedError: None,
                tf.errors.InternalError: None,
                tf.errors.UnavailableError: None,
            },
            propagate={
                optuna.exceptions.TrialPruned: None,
            },
            force_crash_oom=None,
        )

## Main

In [ ]:
# # Search space:
# base_path = f"{RUN_DIR}/search_space/"
# (
#     x_s008_lidar_train,
#     x_s008_lidar_val,
#     x_s008_coord_train,
#     x_s008_coord_val,
#     y_s008_train,
#     y_s008_val,
# ) = train_test_split(
#     s008_lidar_input,
#     s008_coord_input,
#     s008_y_train,
#     test_size=0.2,
#     random_state=0,
#     shuffle=True,
# )


# plot_model_param_distribution(
#     lambda trial: build_model(
#         trial=trial,
#         kparams=kparams,
#         show_summary=False,
#         train_seed=0,
#     ),
#     benchmark_training=False,
#     fit_x=(x_s008_lidar_train, x_s008_coord_train),
#     fit_y=y_s008_train,
#     fit_validation_data=((x_s008_lidar_val, x_s008_coord_val), y_s008_val),
#     bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
#     batch_size=64,
#     n_trials=NUM_TRIALS,
#     fig_save_path=f"{base_path}model_param_distribution.png",
#     csv_path=f"{base_path}model_param_distribution.csv",
#     logs_dir=f"{base_path}logs/",
#     corr_csv_path=f"{base_path}model_param_distribution_corr.csv",
#     # plot_model_dir=f"{base_path}plots/",
#     figsize=(18, 6),
# )

In [ ]:
study = run_study(
    objective=objective,
    run_dir=RUN_DIR,
    num_trials=NUM_TRIALS,
    sampler_seed=SAMPLER_SEED,
    direction=DIRECTION,
    top_k=TOP_K,
    rank_key=RANK_KEY,
    order=ORDER,
    convergence_epoch_column="train_loss",
    convergence_epoch_direction="minimize",
    init_study_dirs=[
        "args",
        "fig",
        "backup",
        "history",
        "scaler",
        "model",
        "logs",
        "tensorboard",
    ],
    cleanup_paths=[
        ("model", "trial_{trial_id}.keras"),
        ("fig", "trial_{trial_id}.png"),
        ("history", "trial_{trial_id}.csv"),
        ("tensorboard", "trial_{trial_id}"),
        ("scaler", "trial_{trial_id}.pkl"),
    ],
    rename_paths=[
        ("model", ".keras"),
        ("fig", ".png"),
        ("history", ".csv"),
        ("scaler", ".pkl"),
    ],
    extra_attrs=[
        "best_epoch",
        "best_train_loss",
        "best_val_loss",
        "s008_loss",
        "s009_loss",
        "best_train_accuracy",
        "best_val_accuracy",
        "s008_accuracy",
        "s009_accuracy",
        "s008_nlos_loss",
        "s009_nlos_loss",
        "s008_nlos_accuracy",
        "s009_nlos_accuracy",
    ],
    variance_threshold=1e-10,
    prune_threshold=50,
    patience=100,
)